In [1]:
# python version
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone
from DataLoader import DataLoader
from DataSplitter import DataSplitter
from PipelineBuilder import Pipeline
from CrossValidation import CrossValidation

In [3]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# pytorch threads
n_threads = os.cpu_count()
torch.set_num_threads(n_threads)
n_threads

8

In [5]:
## Loading data
data_loader = DataLoader(path_train="../data/train.csv", path_eval="../data/validation.csv", path_test="../data/test.csv")
train, validation, test = data_loader.load()
train.shape, validation.shape, test.shape

((29000, 2), (1000, 2), (10000, 2))

In [6]:
## Splitting data
data_splitter = DataSplitter(src_language='english', target_language='portuguese')
X_train, y_train = data_splitter.split(train)
X_val, y_val = data_splitter.split(validation)
X_test, y_test = data_splitter.split(test)

## LSTM

In [7]:
# DL Pipeline 
params = {'model_name': "LSTM", 'tokenizer_method': "sentencepiece", 'vocab_size': 3000, 'max_length': 20, 'embedding_dim': 128, 'hidden_dim': 128, 
          'num_layers': 1, 'dropout': 0.3, 'teacher_forcing': 0.5, 'max_length_decoded': 20, 'lr': 1e-3, 'epochs': 1, 'batch_size': 128} 
params['device'] = "gpu"

pipeline = Pipeline(**params)
pipeline

,max_length,20
,vocab_size,3000
,embedding_dim,128
,hidden_dim,128
,num_layers,1
,dropout,0.3
,teacher_forcing,0.5
,max_length_decoded,20
,lr,0.001
,epochs,1
,batch_size,128


In [8]:
# Cross-Validation
cv = CrossValidation(pipeline=clone(pipeline))

# Holdout Validation
#cv_score = cv.evaluate_one_fold(X_train, y_train, X_test, y_test)
#print("Holdout Validation score: ", cv_score)

In [9]:
## Hyper-parameter tunning
cv = CrossValidation(pipeline=clone(pipeline))

param_grid = {
    'tokenizer_method': ["sentencepiece"],
    'max_length': [30],
    'vocab_size': [5000],
    'max_length_decoded': [30],
    'embedding_dim': [512], 
    'hidden_dim': [512], 
    'num_layers': [2],
    'teacher_forcing': [0.7], 
    'lr': [7e-4], 
    'epochs': [1], 
    'batch_size': [128]
}

cv_lstm_results, loss_curves, _ = cv.hyper_param_tune_one_fold(X_train, y_train, X_test, y_test, param_grid, 
                                                               reduced=True, train_size=10000, return_loss_curves=True)
cv_lstm_results

Time cleaning:  1.0369198322296143
SentencePiece
8.0641
[ 8. 12. 13. 17.]
SentencePiece
7.835
[ 7. 11. 13. 17.]
Time tokenizing:  1.972837209701538
Training Activated ? True
Loss after Epoch 1: 5.7937.
forward time: 13.47656429966446
backward time:  26.990855100215413
loss time:  0.5714516000589356
Loss after gradient descent: 5.793674.
Total Training time: 45.81423568725586.
Params:  {'batch_size': 128, 'embedding_dim': 512, 'epochs': 1, 'hidden_dim': 512, 'lr': 0.0007, 'max_length': 30, 'max_length_decoded': 30, 'num_layers': 2, 'teacher_forcing': 0.7, 'tokenizer_method': 'sentencepiece', 'vocab_size': 5000}
Fit time: 49.09048652648926
SentencePiece
8.3621
[ 8. 12. 14. 18.]
time cleaning and encoding test:  0.8178014755249023
Training Activated ? False
time predicting test:  12.305927991867065
Time decodig test:  0.06385993957519531
Prediction time: 13.187589406967163

Total CV time: 63.42158579826355



,batch_size,embedding_dim,epochs,hidden_dim,lr,max_length,max_length_decoded,num_layers,teacher_forcing,tokenizer_method,vocab_size,fit_time,pred_time,final_loss,final_validation_score,score
0,128,512,1,512,0.0007,30,30,2,0.7,sentencepiece,5000,49.090487,13.187589,5.793674,NaN,0.061827


In [10]:
cv.pipeline.set_params(epochs=150)
cv.pipeline.fit(X_train, y_train)
cv.pipeline.save("translatorLSTM2.pt")

Time cleaning:  3.066371202468872
SentencePiece
8.153413793103448
[ 8. 12. 13. 17.]
SentencePiece
7.920172413793104
[ 7. 12. 13. 17.]
Time tokenizing:  4.024739980697632
Training Activated ? True
Loss after Epoch 1: 5.2408.
Loss after Epoch 6: 2.8738.
Loss after Epoch 11: 1.8867.
Loss after Epoch 16: 1.2084.
Loss after Epoch 21: 0.7960.
Loss after Epoch 26: 0.5150.
Loss after Epoch 31: 0.3614.
Loss after Epoch 36: 0.2805.
Loss after Epoch 41: 0.2225.
Loss after Epoch 46: 0.1948.
Loss after Epoch 51: 0.1800.
Loss after Epoch 56: 0.1723.
Loss after Epoch 61: 0.1492.
Loss after Epoch 66: 0.1473.
Loss after Epoch 71: 0.1342.
Loss after Epoch 76: 0.1376.
Loss after Epoch 81: 0.1268.
Loss after Epoch 86: 0.1263.
Loss after Epoch 91: 0.1214.
Loss after Epoch 96: 0.1132.
Loss after Epoch 101: 0.1158.
Loss after Epoch 106: 0.1102.
Loss after Epoch 111: 0.1130.
Loss after Epoch 116: 0.1072.
Loss after Epoch 121: 0.1069.
Loss after Epoch 126: 0.1104.
Loss after Epoch 131: 0.1106.
Loss after Epoch

In [11]:
cv.pipeline.score(X_test, y_test)

SentencePiece
8.2629
[ 8. 12. 14. 18.]
time cleaning and encoding test:  2.8828396797180176
Training Activated ? False
time predicting test:  13.982325315475464
Time decodig test:  0.27626466751098633


26.0218011350616

In [14]:
preds = cv.pipeline.predict(X_test)
preds_df = pd.DataFrame({'english': X_test.iloc[:, 0].values, 'portuguese': y_test.iloc[:, 0], 'translation': preds})
preds_df.tail(50)

SentencePiece
8.2629
[ 8. 12. 14. 18.]
time cleaning and encoding test:  0.885495662689209
Training Activated ? False
time predicting test:  12.2694673538208
Time decodig test:  0.08709025382995605


,english,portuguese,translation
9950,I have a lot of homework.,Eu tenho muito dever de casa.,Eu tenho muita lição de casa.
9951,This has to be stopped.,Isso tem que ser barrado.,Isso deve ser de.
9952,I do hope so.,Eu realmente espero.,Eu espero que pare.
9953,Tom doesn't like Mary very much.,Tom não gosta muito de Maria.,Tom não gosta muito de Maria.
9954,Are you worried about global warming?,Você está preocupado com o aquecimento global?,Você está preocupado com com pona?
9955,Tom is big and strong.,Tom é grande e forte.,Tom é pequeno e meu peixe.
9956,Tom can draw a perfect circle.,Tom sabe desenhar um círculo perfeito.,Tom pode ser um um jor de.
9957,These children are my grandchildren.,Estas crianças são meus netos.,Esses são meus são estão inidos.
9958,I don't like that part of town.,Não gosto daquela parte da cidade.,Eu não gosto daquele das minhas sorte.
9959,He led a life of luxury.,Ele levou uma vida de luxo.,Ele contgamente em fatos.
